In [2]:
import open3d as o3d
import numpy as np
import time 
import trimesh
from PCAClass import PCABounding, Best_Fit_CPP

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
def slice_mesh_with_plane(mesh, plane_origin, plane_normal, local_faces=None):
    """Slices a mesh with a plane and returns the intersection points."""
    # Convert Open3D mesh to Trimesh
    s=time.time()
    trimesh_mesh = trimesh.Trimesh(vertices=np.asarray(mesh.vertices), faces=np.asarray(mesh.triangles))

    # Slice the mesh with a plane
    slice_result = trimesh.intersections.mesh_plane(trimesh_mesh, plane_normal, plane_origin, local_faces=local_faces)
    e=time.time()
    print(f"Trimesh time=: {e-s} seconds")
    if len(slice_result)==0:
        return None
    return  slice_result 

def calculate_straight_line_distance(points):
    #Calculate the max Euclidean distance between two points in intersection contour
    if len(points) < 2:
        return 0    
    dists = np.linalg.norm(points[:, None] - points, axis=2)  # Pairwise distances
    return np.max(dists)  # Max distance between slice points

def calculate_total_distance_L2(points):
    #Calculate the total Euclidean distance along the intersection contour
    if points is None or len(points) < 2:
        return 0
    diffs=np.diff(points, axis=1)
    print(np.shape(diffs))
    distances=np.linalg.norm(diffs, axis=2)

    return np.sum(distances)  # Sum of all segment distances

def visualize_mesh_with_points(mesh, points):
    mesh.compute_vertex_normals()
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(points[10:50])
    point_cloud.paint_uniform_color([1, 0, 0])  # Red
    o3d.visualization.draw_geometries([mesh, point_cloud],mesh_show_back_face=True)
    return

#mesh = o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
mesh = o3d.io.read_triangle_mesh(r"plane_segments\Non-planar.stl")

# Define slicing plane (e.g., Z = 0 plane)
#plane_origin = np.array([0, 0, 0])  # Origin
plane_normal = np.array([1, 0, 0]) 
print(np.asarray(mesh.vertices))
plane_origin = mesh.vertices[np.random.choice(np.asarray(mesh.vertices).shape[0])]
 

# Slice the mesh
s=time.time()
#relevant_faces= filter_points_in_region(mesh, plane_origin+np.array([-50,-200,-100]), plane_origin+np.array([50,200,100]))
slice_result  = slice_mesh_with_plane(mesh, plane_origin, plane_normal)
e=time.time()
print(f"slice time=: {e-s} seconds")
#print(slice_result)
points=np.unique(np.vstack(slice_result), axis=0)


# Compute distances
if slice_result  is not None:
    straight_line_dist = calculate_straight_line_distance(points)
    real_distance = calculate_total_distance_L2(slice_result)
    visualize_mesh_with_points(mesh, points)
    print(f"Straight-line distance between slice points: {straight_line_dist}")
    print(f"Sum of Euclidean distances between slice points: {real_distance}")
else:
    print("No intersection found.")


[[-400.          721.12927246  100.70954132]
 [-400.          720.04699707  101.73659515]
 [-398.50186157  721.12927246  100.70954132]
 ...
 [ 395.3125      472.21166992  101.73753357]
 [ 395.3125      473.31436157  102.76663208]
 [ 396.875       472.21166992  101.73753357]]
Trimesh time=: 0.06802511215209961 seconds
slice time=: 0.06907415390014648 seconds
(184, 1, 3)
Straight-line distance between slice points: 249.99993896666274
Sum of Euclidean distances between slice points: 276.0342006268062


In [ ]:
#%%timeit

#Assumes contiguous segments (slips would be segmented out)
def find_max_curvilinear_dist(plane_normal, plane_origin, step):
    max_distance = 0.0
    intersection = True
    
    while intersection:
        slice_result = trimesh.intersections.mesh_plane(trimesh_mesh, plane_normal, plane_origin)
        
        if len(slice_result) == 0:
            intersection = False
            #print("No Intersection")
        else:
            points=np.unique(np.vstack(slice_result), axis=0)
            # Sort points along the secondary axis
            sorted_indices = np.argsort(points @ segment.secondary_axis)
            points = points[sorted_indices]

            # Compute distances
            diffs = np.diff(points, axis=0)
            distances = np.linalg.norm(diffs, axis=1)
            total_distance = np.sum(distances)  # Total length of the intersection curve

            max_distance = max(max_distance, total_distance)
            plane_origin=plane_origin +step * plane_normal

    # point_cloud = o3d.geometry.PointCloud()
    # point_cloud.points = o3d.utility.Vector3dVector(points[10:125])
    # point_cloud.paint_uniform_color([1, 0, 0])  # Red
    # o3d.visualization.draw_geometries([mesh, point_cloud],mesh_show_back_face=True)

    return max_distance

#segment=Best_Fit_CPP(r"plane_segments\Non-planar.stl")
segment=Best_Fit_CPP(r"plane_segments\Hyper_meshed_noise.stl")
segment.mesh.compute_adjacency_list()
segment.mesh.compute_triangle_normals()
segment.mesh.compute_vertex_normals()


plane_origin=segment.bounding_box.center
plane_normal=segment.primary_axis
trimesh_mesh = trimesh.Trimesh(vertices=np.asarray(segment.mesh.vertices), faces=np.asarray(segment.mesh.triangles))

max_forward_distance = find_max_curvilinear_dist(plane_normal, plane_origin, step=10)
max_backward_distance = find_max_curvilinear_dist(plane_normal, plane_origin, step=-10)

# **Final result: maximum distance across all slices**
max_distance = max(max_forward_distance, max_backward_distance)


Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
Maximum intersection distance: 247.60 mm
444 ms ± 9.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
